# P4-1 · Bài 4 · Làm việc với LLM

Toàn bộ code của bài, chạy từ trên xuống như trên trang bài học. Kết quả đã chạy sẵn được giữ trong notebook; bấm **Runtime → Run all** để chạy lại. Ô có dòng đầu `# Cần GPU` thì đổi **Runtime → Change runtime type → T4 GPU**; ô cần API key thì đặt khóa trong Colab Secrets.

In [ ]:
!pip install -q anthropic

## Lý thuyết

In [ ]:
# Cần ANTHROPIC_API_KEY: chạy trên máy bạn hoặc Colab (gọi API thật, mất phí theo token)
import anthropic
client = anthropic.Anthropic()                       # đọc ANTHROPIC_API_KEY từ biến môi trường
resp = client.messages.create(
    model="claude-haiku-5-5",
    max_tokens=1024,
    system="Bạn là trợ lý chăm sóc khách hàng của một shop nội thất. Trả lời ngắn gọn, lịch sự.",
    messages=[{"role": "user", "content": "Bàn trang điểm có kèm gương không?"}],
)
print(resp.content[0].text)
print(resp.stop_reason, resp.usage.input_tokens, resp.usage.output_tokens)

In [ ]:
from pydantic import BaseModel, ValidationError, Field
class Invoice(BaseModel):
    vendor: str
    date: str = Field(description="YYYY-MM-DD")
    total_vnd: int
    items: list[str]

def extract(text, retries=2):
    msg = (f"Trích xuất hóa đơn thành JSON đúng schema sau, chỉ trả JSON:\n{Invoice.model_json_schema()}\n"
           f"<document>{text}</document>")
    history = [{"role": "user", "content": msg}]
    for _ in range(retries + 1):
        out = client.messages.create(model="claude-haiku-5-5", max_tokens=800, messages=history).content[0].text
        try:
            return Invoice.model_validate_json(out.strip().strip("`").removeprefix("json"))   # bỏ khung code nếu có
        except ValidationError as e:
            history += [{"role": "assistant", "content": out},
                        {"role": "user", "content": f"JSON chưa hợp lệ: {e}. Sửa và chỉ trả JSON."}]
    raise ValueError("không trích xuất được")

# phần kiểm tra schema chạy được ngay, không cần gọi API:
ok = Invoice.model_validate_json('{"vendor": "Nội thất An Phát", "date": "2026-10-02", "total_vnd": 1500000, "items": ["bàn trang điểm"]}')
print(ok.total_vnd, ok.items)
try:
    Invoice.model_validate_json('{"vendor": "An Phát", "total_vnd": "một triệu rưỡi"}')
except ValidationError as e:
    print(e.error_count(), "lỗi ở các trường:", [err["loc"][0] for err in e.errors()])   # lỗi này được gửi lại cho model

1500000 ['bàn trang điểm']
3 lỗi ở các trường: ['date', 'total_vnd', 'items']


In [ ]:
# Cần ANTHROPIC_API_KEY và client ở ô đầu: chạy trên máy bạn hoặc Colab
tools = [{
    "name": "get_order_status",
    "description": "Tra trạng thái đơn hàng theo mã đơn. Dùng khi khách hỏi đơn của họ đang ở đâu hoặc khi nào tới.",
    "input_schema": {"type": "object",
                     "properties": {"order_id": {"type": "string", "description": "Mã đơn, dạng DH + số, ví dụ DH1024"}},
                     "required": ["order_id"]},
}]
messages = [{"role": "user", "content": "Đơn DH1024 của tôi tới đâu rồi?"}]
resp = client.messages.create(model="claude-haiku-5-5", max_tokens=500, tools=tools, messages=messages)
while resp.stop_reason == "tool_use":
    messages.append({"role": "assistant", "content": resp.content})
    results = []
    for call in [b for b in resp.content if b.type == "tool_use"]:      # có thể nhiều lời gọi cùng lúc
        out = TOOLS[call.name](**call.input)                            # TOOLS: dict tên → hàm Python của bạn
        results.append({"type": "tool_result", "tool_use_id": call.id, "content": str(out)})
    messages.append({"role": "user", "content": results})
    resp = client.messages.create(model="claude-haiku-5-5", max_tokens=500, tools=tools, messages=messages)
print(resp.content[0].text)

In [ ]:
# Cần ANTHROPIC_API_KEY: chạy trên máy bạn hoặc Colab
with client.messages.stream(model="claude-haiku-5-5", max_tokens=1024, messages=messages) as s:
    for text in s.text_stream:
        print(text, end="", flush=True)

In [ ]:
import random, time
def call_with_retry(fn, max_tries=6):
    for k in range(max_tries):
        try:
            return fn()
        except anthropic.APIStatusError as e:
            retryable = e.status_code == 429 or e.status_code >= 500     # rate limit hoặc lỗi phía máy chủ
            if not retryable or k == max_tries - 1:
                raise                                                     # lỗi 400 (yêu cầu sai) thì không thử lại
        except anthropic.APIConnectionError:                              # mất mạng, timeout
            if k == max_tries - 1:
                raise
        time.sleep(min(30, 2 ** k) * random.uniform(0.5, 1.0))

# thử với một hàm giả lập: hai lần đầu bị rate limit (429), lần ba thành công
import anthropic, httpx
calls = {"n": 0}
def flaky():
    calls["n"] += 1
    if calls["n"] < 3:
        raise anthropic.RateLimitError("rate limited", response=httpx.Response(429, request=httpx.Request("POST", "https://api.anthropic.com")), body=None)
    return "ok"
t0 = time.time()
print(call_with_retry(flaky), "sau", calls["n"], "lần gọi,", round(time.time() - t0, 1), "giây chờ")

ok sau 3 lần gọi, 2.2 giây chờ


## Bài tập 4: Tỉ lệ thành công của trích xuất {code}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
# Cần ANTHROPIC_API_KEY và file invoices_eval.json của bạn
import json, time

def evaluate(samples):
    ok_json = field_ok = 0
    fields = ["vendor", "date", "total_vnd"]
    t0 = time.time()
    for s in samples:
        try:
            inv = extract(s["text"])
            ok_json += 1
            field_ok += sum(getattr(inv, f) == s["gold"][f] for f in fields)
        except ValueError:
            pass
    n = len(samples)
    print(f"JSON hợp lệ {ok_json / n:.0%} | đúng từng trường {field_ok / (n * len(fields)):.0%} | {(time.time() - t0) / n:.1f}s mỗi mẫu")

samples = json.load(open("invoices_eval.json", encoding="utf-8"))   # [{"text": ..., "gold": {...}}, ...]
evaluate(samples)

## Bài tập 5: Giữ hội thoại trong ngân sách token {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
system = "Bạn là trợ lý chăm sóc khách hàng của shop nội thất. Trả lời ngắn gọn, lịch sự."
messages = []
for i in range(1, 21):
    messages.append({"role": "user", "content": f"Câu hỏi số {i}: đơn hàng của tôi giao tới đâu rồi, mã đơn DH{i:03d}?"})
    messages.append({"role": "assistant", "content": f"Đơn DH{i:03d} đang ở kho Hà Nội, dự kiến giao trong 2 ngày tới ạ."})

def uoc_token(text):
    return len(text) // 3 + 1

def cat_lich_su(messages, ngan_sach):
    # TODO: trả về list các message gần nhất, tổng uoc_token (cộng cả system) <= ngan_sach, bắt đầu bằng user
    return messages

In [ ]:
def cat_lich_su(messages, ngan_sach):
    con_lai = ngan_sach - uoc_token(system)
    giu = []
    for m in reversed(messages):
        t = uoc_token(m["content"])
        if t > con_lai:
            break
        giu.append(m); con_lai -= t
    giu.reverse()
    while giu and giu[0]["role"] != "user":
        giu.pop(0)
    return giu

kept = cat_lich_su(messages, 300)
print(f"giữ {len(kept)}/{len(messages)} message, bắt đầu bằng: {kept[0]['content'][:30]}...")

giữ 12/40 message, bắt đầu bằng: Câu hỏi số 15: đơn hàng của tô...


## Bài tập 6: Chọn ví dụ few-shot phù hợp {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
kho = [("Đơn của tôi 5 ngày chưa giao", "giao_hang"), ("Shipper không gọi trước khi giao", "giao_hang"),
       ("Bàn bị trầy, tôi muốn đổi cái khác", "doi_tra"), ("Muốn trả hàng vì không vừa phòng", "doi_tra"),
       ("Chuyển khoản rồi mà đơn vẫn chưa thanh toán", "thanh_toan"), ("Bị trừ tiền hai lần", "thanh_toan"),
       ("Gương có chống mờ hơi nước không", "san_pham"), ("Bàn này chịu được bao nhiêu kg", "san_pham")]

def tu(s):
    return set(s.lower().split())

def build_prompt(ticket, k=3):
    # TODO: chọn k ví dụ có nhiều từ chung với ticket nhất, dựng prompt
    return ""

ticket = "Tôi chuyển khoản hai lần cho cùng một đơn, đơn vẫn báo chưa thanh toán"

In [ ]:
def build_prompt(ticket, k=3):
    chon = sorted(kho, key=lambda ex: len(tu(ex[0]) & tu(ticket)), reverse=True)[:k]
    vi_du = "\n".join(f"<example><text>{t}</text><label>{l}</label></example>" for t, l in chon)
    return (
        "Bạn là nhân viên phân loại ticket hỗ trợ của một shop nội thất.\n"
        "Nhãn hợp lệ: giao_hang, doi_tra, thanh_toan, san_pham.\n"
        f"Ví dụ:\n{vi_du}\n"
        f"<ticket>{ticket}</ticket>\n"
        "Chỉ trả về đúng một nhãn, không giải thích."
    )

prompt = build_prompt(ticket)
print(prompt)

Bạn là nhân viên phân loại ticket hỗ trợ của một shop nội thất.
Nhãn hợp lệ: giao_hang, doi_tra, thanh_toan, san_pham.
Ví dụ:
<example><text>Chuyển khoản rồi mà đơn vẫn chưa thanh toán</text><label>thanh_toan</label></example>
<example><text>Đơn của tôi 5 ngày chưa giao</text><label>giao_hang</label></example>
<example><text>Bị trừ tiền hai lần</text><label>thanh_toan</label></example>
<ticket>Tôi chuyển khoản hai lần cho cùng một đơn, đơn vẫn báo chưa thanh toán</ticket>
Chỉ trả về đúng một nhãn, không giải thích.


## Bài tập 7: Vòng lặp tool calling {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import json
DON = {"DH001": {"trang_thai": "đang giao", "tinh": "Đà Nẵng", "can_nang": 18}}

def tra_don(ma_don):
    return DON.get(ma_don, {"loi": f"không có đơn {ma_don}"})

def phi_ship(tinh, can_nang):
    return {"phi": 30000 + (2000 * can_nang if tinh != "Hà Nội" else 1000 * can_nang)}

TOOLS = {"tra_don": tra_don, "phi_ship": phi_ship}

def fake_llm(messages):
    """Đóng vai model: lần lượt gọi tra_don, rồi phi_ship, rồi trả lời."""
    ket_qua = [b for m in messages if m["role"] == "user" and isinstance(m["content"], list) for b in m["content"]]
    if len(ket_qua) == 0:
        return {"stop_reason": "tool_use", "content": [{"type": "tool_use", "id": "t1", "name": "tra_don", "input": {"ma_don": "DH001"}}]}
    if len(ket_qua) == 1:
        don = json.loads(ket_qua[0]["content"])
        return {"stop_reason": "tool_use", "content": [{"type": "tool_use", "id": "t2", "name": "phi_ship",
                                                        "input": {"tinh": don["tinh"], "can_nang": don["can_nang"]}}]}
    phi = json.loads(ket_qua[1]["content"])["phi"]
    return {"stop_reason": "end_turn", "content": [{"type": "text", "text": f"Đơn DH001 đang giao tới Đà Nẵng, phí ship {phi:,}đ."}]}

def chay_agent(cau_hoi, max_buoc=5):
    messages = [{"role": "user", "content": cau_hoi}]
    # TODO: lặp gọi fake_llm, xử lý tool_use, trả về (câu trả lời, số lần gọi model)
    return "", 0

In [ ]:
def chay_agent(cau_hoi, max_buoc=5):
    messages = [{"role": "user", "content": cau_hoi}]
    for so_lan in range(1, max_buoc + 1):
        resp = fake_llm(messages)
        messages.append({"role": "assistant", "content": resp["content"]})
        if resp["stop_reason"] == "end_turn":
            return resp["content"][0]["text"], so_lan
        ket_qua = []
        for b in resp["content"]:
            if b["type"] == "tool_use":
                kq = TOOLS[b["name"]](**b["input"])
                print(f"  gọi {b['name']}({b['input']}) → {kq}")
                ket_qua.append({"type": "tool_result", "tool_use_id": b["id"], "content": json.dumps(kq, ensure_ascii=False)})
        messages.append({"role": "user", "content": ket_qua})
    return "Dừng: quá số bước cho phép", max_buoc

tra_loi, so_lan = chay_agent("Đơn DH001 của tôi đến đâu rồi, phí ship bao nhiêu?")
print(tra_loi, f"({so_lan} lần gọi model)")

  gọi tra_don({'ma_don': 'DH001'}) → {'trang_thai': 'đang giao', 'tinh': 'Đà Nẵng', 'can_nang': 18}
  gọi phi_ship({'tinh': 'Đà Nẵng', 'can_nang': 18}) → {'phi': 66000}
Đơn DH001 đang giao tới Đà Nẵng, phí ship 66,000đ. (3 lần gọi model)


## Bài tập 8: Gọi API chịu lỗi: retry có backoff và stream {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import random, time
random.seed(0)
so_lan_goi = 0

class QuaTai(Exception):
    pass

def fake_stream():
    global so_lan_goi
    so_lan_goi += 1
    if so_lan_goi <= 2:
        raise QuaTai("529 overloaded")
    for manh in ["Dạ, ", "đơn của ", "anh/chị ", "sẽ giao ", "trong 2 ngày."]:
        time.sleep(0.05)
        yield manh

def goi_co_retry(max_lan=5):
    # TODO: thử lại khi QuaTai, chờ theo backoff; stream và đo thời gian tới mảnh đầu
    return "", 0, 0.0

In [ ]:
def goi_co_retry(max_lan=5):
    for lan in range(1, max_lan + 1):
        t0 = time.perf_counter(); ttft = None; ket_qua = ""
        try:
            for manh in fake_stream():
                if ttft is None:
                    ttft = time.perf_counter() - t0
                print(manh, end="", flush=True)
                ket_qua += manh
            print()
            return ket_qua, lan, ttft
        except QuaTai as e:
            cho = 0.1 * 2 ** lan * (0.5 + random.random())
            print(f"lần {lan}: {e}, chờ {cho:.2f} giây")
            time.sleep(cho)
    raise RuntimeError("Hết số lần thử")

tra_loi, so_lan_thu, ttft = goi_co_retry()
print(f"thành công ở lần {so_lan_thu}, tới mảnh đầu sau {ttft * 1000:.0f} ms")

lần 1: 529 overloaded, chờ 0.27 giây
lần 2: 529 overloaded, chờ 0.50 giây
Dạ, đơn của anh/chị sẽ giao trong 2 ngày.
thành công ở lần 3, tới mảnh đầu sau 50 ms
